# Ders 16 - Microsoft Foundry ile Ölçeklenebilir Ajanların Dağıtımı

Bu not defterinde, kurgu şirket **Contoso** için **üretime hazır müşteri destek ajanı** inşa edersiniz. Önceki derslerin aksine, burada önemli olan ajanın akıl yürütme döngüsü değil — ölçeklenebilir şekilde güvenli çalışmasını sağlayan *etrafında* olan her şeydir:

1. **Araç çağırma** — sipariş sorgulamaları ve bilet oluşturma.
2. **RAG** — bilgi tabanından politika yanıtları.
3. **Bellek** — müşteriyle yapılan konuşma boyunca müşteri bilgisini hatırlama.
4. **Model yönlendirmesi** — basit istekleri küçük bir modele, karmaşık olanları büyük modele gönderme.
5. **Yanıt önbellekleme** — tekrarlanan soruları model çağrısı olmadan yanıtlama.
6. **İnsan onayı** — belirli bir eşik üzerindeki iade işlemleri onay için duraklatılır.
7. **Değerlendirme kapısı** — kötü bir sürümü engelleyen çevrimdışı test seti.
8. **Gözlemlenebilirlik** — her isteğin etrafında OpenTelemetry izleme.

Her bölüm kendi içinde bağımsızdır ve çalıştırılabilir. Her satırı okuyun — üretim temel öğeleri kasıtlı olarak küçük tutulmuştur.


## Kurulum

Bu defteri çalıştırmadan önce, aşağıdakilere sahip olduğunuzdan emin olun:

1. **Dağıtılmış bir sohbet modeli bulunan bir Microsoft Foundry projesi** (örneğin `gpt-4.1-mini`).
2. **Azure CLI ile giriş yapmış olmak** — terminalinizde `az login` komutunu çalıştırın.
3. **Gerekli ortam değişkenlerini ayarlamak:**
   - `AZURE_AI_PROJECT_ENDPOINT` — Microsoft Foundry projenizin uç noktası.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — dağıtılmış modelinizin adı.

RAG bölümü, `AZURE_SEARCH_SERVICE_ENDPOINT` ve `AZURE_SEARCH_API_KEY` ayarlandığında **Azure AI Search** kullanır ve bu değişkenler yoksa, defterin Search kaynağı olmadan çalışabilmesi için bellekte arama yöntemine döner.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. Araçlar

Üretim araçları gerçek sistemlere karşı gerçek işler yapar. Burada basit Python fonksiyonları ile bir sipariş veritabanını ve bir biletleme sistemini simüle ediyoruz. `@tool` dekoratörü bunları ajan için erişilebilir kılar.

Dikkat edin, `issue_refund` geri ödemeler için bir eşik üzerinde `approval_mode="always_require"` kullanıyor — bu, daha sonra uygulayacağımız insan-döngüde ilkelidir.


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — Politika Bilgi Tabanı

Politika sorularına ("iade süreniz nedir?") modelin belleğinden değil, yetkili bir kaynaktan cevap verilmelidir. Küçük bir bilgi tabanını arama aracı olarak kullanıyoruz.

Üretimde bu **Azure AI Search**; burada defterin her yerde çalışması için bellek içi anahtar kelime araması sunuyoruz ve ortam değişkenleri mevcut olduğunda otomatik olarak Azure AI Search'e geçiyoruz.


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. Bellek

Kiminle konuştuğunu unutan bir destek temsilcisi kötü bir destek temsilcisidir. Müşteri başına küçük bir profil deposu tutarız ve kısa bir özeti temsilcinin talimatlarına ekleriz. Üretimde bu bir bellek servisidir (Bkz. Ders 13); burada bir sözlük modeli görünür kılar.


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 & 5. Model Yönlendirme ve Yanıt Önbellekleme

Tek bir istek işleyicisine bağlı iki maliyet kaldıraçları:

- **Yönlendirme**: ucuz bir sezgisel sınıflandırıcı, bir isteğin küçük mü yoksa büyük model mi gerektirdiğine karar verir.
- **Önbellekleme**: normalleştirilmiş tekrar eden sorular doğrudan model çağrısı olmadan bir önbellekten sunulur.

Buradaki sınıflandırıcı kasıtlı olarak basittir. Üretimde bunu trafiğe karşı doğrularsınız ve Foundry'nin Model Yönlendiricisi ile değiştirebilirsiniz.


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-4.1-mini
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-4.1

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. Ajan, İnsan Onayı ve Gözlemlenebilirlik

Şimdi yukarıdaki araçlardan ajanı bir araya getiriyoruz ve her isteği bir OpenTelemetry span içinde sarıyoruz. `handle_support_request` fonksiyonu üretim istek işleyicisidir: önbellek → yönlendirme → izleme → çalışma → önbellek.

İnsan onayı framwork tarafından yönetilir: çünkü `issue_refund` `approval_mode="always_require"` olarak ayarlandığından, çalışma duraklar ve açıkça çözdüğümüz bir onay isteği gösterir.


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

support_agent = provider.as_agent(
    name="ContosoSupportAgent",
    instructions=SUPPORT_INSTRUCTIONS,
    tools=[get_order_status, open_ticket, search_policies, issue_refund],
)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await support_agent.run(prompt, model=chosen_model)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. Değerlendirme Kapısı

Bu, dersten çıkan sürüm kapısıdır: çevrimdışı bir test seti ajanı puanlar ve dağıtım, geçme oranı bir eşik değerini aşarsa devam eder. Buradaki puanlayıcı, defteri kendi içinde tutmak için basit bir anahtar kelime örtüşme kontrolüdür; üretimde bir LLM-jüri ya da bir çerçeve değerlendirmesi kullanırsınız (bkz. Ders 10).


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## Birleştirmek: Simüle Edilmiş Bir Yayın

Aşağıdaki hücre, dersin anlattığı tüm döngüyü gösterir: değerlendirme kapısını çalıştırın ve ancak geçerse "yayınlayın". Bu, bir ajan sürümünü Foundry Agent Service'e terfi ettirmeden önce CI'da çalıştıracağınız kalıptır.


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## Özet

Operasyonel tüm endişeler entegre edilmiş, üretime hazır bir müşteri destek temsilcisi oluşturduğunuz:

- **Araçlar, RAG ve bellek**, temsilciye yetenek ve bağlam sağlar.
- **Model yönlendirmesi ve önbellekleme**, gecikme ve maliyeti kontrol altında tutar.
- **İnsan onayı**, büyük iadeler gibi yüksek riskli işlemleri korur.
- **Değerlendirme kapısı**, kötü sürümleri yayına alınmadan engeller.
- **İzleme**, her isteğin gözlemlenmesini sağlar.

### Zorluk

Bu temsilciyi genişletin:

1. **Birden fazla modeli destekleyin** — üçüncü bir "akıl yürütme" katmanı ekleyin ve yükseltmeleri/şikayetleri ona yönlendirin.
2. **Değerlendirme kapıları ekleyin** — `TEST_CASES`i, iade onayı senaryolarını da içerecek şekilde genişletin ve kapının gerilemeyi yakaladığından emin olun.
3. **Maliyet bilincine sahip yönlendirme ekleyin** — istek başına tahmini bir maliyet (küçük, büyük, önbellek) takip edin ve karışık sorgulardan oluşan bir grup işlemden sonra bir maliyet raporu yazdırın.

Bir sonraki derste, tam ters yolu izleyip bir temsilciyi tamamen kendi makinenizde Microsoft Foundry Local ve Qwen ile çalıştıracaksınız.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba sarf etsek de, otomatik çevirilerin hata veya yanlışlık içerebileceğini lütfen unutmayınız. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek yanlış anlamalardan veya yanlış yorumlamalardan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
